# 🔬 Feature Lab – Quant Researcher
**Workflow**: Load OHLCV → Compute features → Interactive EDA → Feature importance → Correlation Analysis → Save to FeatureStore

---

In [ ]:
from qtrader.research import AnalystSession, RoleContext
from qtrader.features.store import FeatureStore
import polars as pl
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

session = AnalystSession(role=RoleContext.RESEARCHER)
session.info()

SYMBOL = 'BTC-USD'
TIMEFRAME = '1d'

## 1. Data Ingestion & Engineering
Load market data and augment with technical factors.

In [ ]:
try:
    df = session.load_ohlcv(SYMBOL, TIMEFRAME, days=365)
except Exception:
    df = session.sample_ohlcv(symbol='BTC', days=365)

df = session.make_returns(df)
df = session.add_rolling_features(df, windows=[5, 10, 21, 63])
df = df.drop_nulls()

# Alpha scores (forward returns)
df = session.run_alpha_score(df, forward_periods=[1, 5, 10])
df = df.drop_nulls()

print(f"Dataset shape: {df.shape}")
df.head()

## 2. Interactive Correlation Analysis
Examine how features correlate with forward returns across different horizons.

In [ ]:
feature_cols = [c for c in df.columns if c.startswith(('sma_', 'vol_', 'rsi_', 'avg_'))]
target_cols  = ['fwd_ret_1', 'fwd_ret_5', 'fwd_ret_10']
df = df.drop_nulls()

corr_matrix = []
for fc in feature_cols:
    row = []
    for tc in target_cols:
        c = np.corrcoef(df[fc], df[tc])[0, 1]
        row.append(c)
    corr_matrix.append(row)

fig = px.imshow(
    corr_matrix, 
    x=target_cols, 
    y=feature_cols, 
    color_continuous_scale='RdBu_r', 
    aspect="auto",
    labels=dict(color="Correlation"),
    title="Feature × Forward Return Correlation Matrix"
)
fig.update_layout(
    template="plotly_dark",
    plot_bgcolor='#0f1117',
    paper_bgcolor='#0f1117',
    width=800, 
    height=max(400, len(feature_cols)*25)
)
fig.show()

## 3. Dynamic Feature Stability
Check if the predictive power of the top feature is stable over time.

In [ ]:
corr_vals = np.array(corr_matrix)
top_idx = np.unravel_index(np.argmax(np.abs(corr_vals)), corr_vals.shape)
top_feat = feature_cols[top_idx[0]]
target_feat = target_cols[top_idx[1]]

print(f"Analyzing Stability for: {top_feat} vs {target_feat}")

win = 90
timestamps = df['timestamp'].to_numpy()[win:]
feat_vals = df[top_feat].to_numpy()
target_vals = df[target_feat].to_numpy()

rolling_corr = [np.corrcoef(feat_vals[i:i+win], target_vals[i:i+win])[0,1] for i in range(len(feat_vals)-win)]

fig_stab = go.Figure()
fig_stab.add_trace(go.Scatter(
    x=timestamps, 
    y=rolling_corr, 
    mode='lines', 
    name=f'Rolling {win}d Corr',
    line=dict(color='#a78bfa', width=2)
))
fig_stab.add_hline(y=0, line_dash="dash", line_color="#64748b")
fig_stab.update_layout(
    title=f"Predictive Stability: {top_feat} vs {target_feat}",
    template="plotly_dark",
    plot_bgcolor='#0f1117',
    paper_bgcolor='#0f1117',
    xaxis_title="Date",
    yaxis_title="Correlation",
    height=400
)
fig_stab.show()

## 4. Feature Distribution & Outliers
Understanding feature behavior across different market conditions.

In [ ]:
fig_dist = px.histogram(
    df.to_pandas(), 
    x=top_feat, 
    marginal="box", 
    title=f"Distribution of {top_feat}", 
    color_discrete_sequence=['#34d399']
)
fig_dist.update_layout(
    template="plotly_dark",
    plot_bgcolor='#0f1117',
    paper_bgcolor='#0f1117'
)
fig_dist.show()

## 5. Persistence
Save engineered features to the unified store.

In [ ]:
store = FeatureStore()
save_cols = ['timestamp', 'close', 'returns'] + feature_cols + target_cols
store.save_features(df.select(save_cols), symbol=SYMBOL, timeframe=TIMEFRAME)
print(f"✅ Features persisted to store for {SYMBOL}/{TIMEFRAME}")